In [14]:
import sys
sys.path.append('/home/pulpo/Documents/quantitative-trading')
from src.quanttrading.data import prepare_data
from backtesting import Backtest, Strategy
import pandas as pd

In [15]:
# DOWNLOAD DATA
df_aapl_full = prepare_data('AAPL', '1y', '4h')

[*********************100%***********************]  1 of 1 completed


In [16]:
# SPLIT DATA (TRAIN / TEST)
split_point = len(df_aapl_full) // 2

df_train = df_aapl_full.iloc[:split_point].copy()
df_test = df_aapl_full.iloc[split_point].copy()

print(f'Train: {df_train.index[0]} a {df_train.index[-1]}')
print(f'Test: {df_test.index[0]}, a {df_test.index[-1]}')

Train: 2025-05-02 17:30:00+00:00 a 2025-10-29 17:30:00+00:00
Test: Close, a Return


In [17]:
class AAPL_Strategy(Strategy):
    sl_pct = 0.015
    tp_pct = 0.04
    window = 30

    def init(self):
        close = pd.Series(self.data.Close)
        returns = close.pct_change() * 100

        self.rolling_std = self.I(lambda x: pd.Series(x).rolling(self.window).std(), close)

    def next(self):
        if self.position:
            return
        
        price = self.data.Close[-1]

        if self.rolling_std > 2.3:
            self.open_long(price)

    def open_long(self, price):
        self.buy(
            sl=price * (1 - self.sl_pct),
            tp=price * (1 + self.tp_pct)
        )

In [18]:
# Backtest en TRAIN
print("\n" + "="*60)
print("TRAIN SET")
print("="*60)
bt_train = Backtest(df_train, AAPL_Strategy, cash=100000, commission=0.0001)
results_train = bt_train.run()
print(results_train)

# Backtest en TEST
print("\n" + "="*60)
print("TEST SET (Datos NO vistos)")
print("="*60)
bt_test = Backtest(df_test, AAPL_Strategy, cash=100000, commission=0.0001)
results_test = bt_test.run()
print(results_test)

# COMPARACIÓN
print("\n" + "="*60)
print("COMPARACIÓN TRAIN vs TEST")
print("="*60)
comparison = pd.DataFrame({
    'Train': [results_train['Return [%]'], results_train['Win Rate [%]'], results_train['Expectancy [%]']],
    'Test': [results_test['Return [%]'], results_test['Win Rate [%]'], results_test['Expectancy [%]']]
}, index=['Return %', 'Win Rate %', 'Expectancy %'])
print(comparison)


TRAIN SET
Start                     2025-05-02 17:30...
End                       2025-10-29 17:30...
Duration                    180 days 00:00:00
Exposure Time [%]                    82.66129
Equity Final [$]                 140447.33338
Equity Peak [$]                  140551.38733
Commissions [$]                     553.11451
Return [%]                           40.44733
Buy & Hold Return [%]                37.18341
Return (Ann.) [%]                    98.32945
Volatility (Ann.) [%]                37.08556
CAGR [%]                             60.88624
Sharpe Ratio                          2.65142
Sortino Ratio                        10.51487
Calmar Ratio                         21.52785
Alpha [%]                            17.52329
Beta                                  0.61651
Max. Drawdown [%]                    -4.56755
Avg. Drawdown [%]                    -1.58783
Max. Drawdown Duration       26 days 04:00:00
Avg. Drawdown Duration        6 days 10:00:00
# Trades               

/tmp/ipykernel_17679/854280681.py:6: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  results_train = bt_train.run()


TypeError: `data` must be a pandas.DataFrame with columns